# CardioScope AI - Clinical Model Training & Performance Pipeline
End-to-end multimodal machine learning pipeline for coronary artery disease (CAD) and vessel-specific stenosis (LAD, LCX, RCA) with domain feature engineering, zero-target-leakage validation, TreeSHAP feature attributions, and deep learning benchmarking.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Any

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix, roc_curve
)
import shap
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Global reproducibility seed
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Runtime Acceleration:", f"Hardware Accelerated ({device})" if torch.cuda.is_available() else "Standard CPU")
print("PyTorch Version:", torch.__version__)
print("Environment initialized successfully.")


Runtime Acceleration: Hardware Accelerated (cuda:0)
PyTorch Version: 2.10.0+cu130
Environment initialized successfully.


In [2]:
DATA_CANDIDATE_PATHS = [
    "backend/data/raw/extention of Z-Alizadeh sani dataset.xlsx",
    "data/raw/extention of Z-Alizadeh sani dataset.xlsx",
    "../backend/data/raw/extention of Z-Alizadeh sani dataset.xlsx"
]
RAW_DATA_PATH = next((p for p in DATA_CANDIDATE_PATHS if os.path.exists(p)), None)
if RAW_DATA_PATH is None:
    raise FileNotFoundError("Raw dataset not found in candidate paths.")

print(f"Loading dataset from: {RAW_DATA_PATH}")
df_raw = pd.read_excel(RAW_DATA_PATH)
print(f"Cohort Size: {df_raw.shape[0]} patients, {df_raw.shape[1]} clinical variables")

targets_raw = {
    'CAD (Cath)': (df_raw['Cath'].astype(str).str.strip().str.upper() == 'CAD').sum(),
    'LAD Stenosis': (df_raw['LAD'].astype(str).str.strip().str.capitalize() == 'Stenotic').sum(),
    'LCX Stenosis': (df_raw['LCX'].astype(str).str.strip().str.capitalize() == 'Stenotic').sum(),
    'RCA Stenosis': (df_raw['RCA'].astype(str).str.strip().str.capitalize() == 'Stenotic').sum(),
}
print("\nTarget Prevalence in Cohort:")
for name, count in targets_raw.items():
    print(f"  - {name:<15}: {count:3d}/{len(df_raw)} positive ({count/len(df_raw)*100:5.1f}%)")


Loading dataset from: backend/data/raw/extention of Z-Alizadeh sani dataset.xlsx
Cohort Size: 303 patients, 59 clinical variables

Target Prevalence in Cohort:
  - CAD (Cath)     : 216/303 positive ( 71.3%)
  - LAD Stenosis   : 177/303 positive ( 58.4%)
  - LCX Stenosis   : 119/303 positive ( 39.3%)
  - RCA Stenosis   : 114/303 positive ( 37.6%)


In [3]:
def add_domain_features(df_input: pd.DataFrame) -> pd.DataFrame:
    """
    Computes deterministic physiological domain features from clinical inputs.
    Row-wise transformation with zero target leakage.
    """
    df_out = df_input.copy()
    
    # 1. Lipid Atherogenic Ratio (LDL / HDL)
    df_out['Cholesterol_Ratio'] = df_out['LDL'] / (df_out['HDL'] + 1e-5)
    
    # 2. Metabolic Syndrome Index (FBS * BMI)
    bmi = df_out['BMI'] if 'BMI' in df_out.columns and not df_out['BMI'].isnull().all() else (df_out['Weight'] / ((df_out['Length']/100)**2))
    df_out['Metabolic_Index'] = (df_out['FBS'] * bmi) / 100.0
    
    # 3. Hemodynamic Index (BP / PR)
    df_out['Hemodynamic_Ratio'] = df_out['BP'] / (df_out['PR'] + 1e-5)
    
    # 4. Composite Ischemic Burden (Angina + ST + T-wave)
    angina = (df_out['Typical Chest Pain'] == 1).astype(float)
    st_dep = (df_out['St Depression'] == 1).astype(float)
    st_elev = (df_out['St Elevation'] == 1).astype(float)
    t_inv = (df_out['Tinversion'] == 1).astype(float)
    df_out['Ischemic_Burden'] = (angina * 2.0) + (st_dep * 1.5) + (st_elev * 2.0) + (t_inv * 1.0)
    
    # 5. Ventricular Performance Index (EF-TTE / (RWMA + 1))
    rwma = df_out['Region RWMA'].fillna(0).astype(float)
    df_out['Ventricular_Perf_Index'] = df_out['EF-TTE'] / (rwma + 1.0)
    
    # 6. Vascular Inflammatory Index (ESR * Age / 100)
    df_out['Inflammatory_Age_Index'] = (df_out['ESR'] * df_out['Age']) / 100.0
    
    return df_out

df_engineered = add_domain_features(df_raw)

# Strict Leakage Isolation
leakage_cols = ['LAD', 'LCX', 'RCA', 'Cath']
X_df = df_engineered.drop(columns=leakage_cols)

targets = {
    'CAD': (df_raw['Cath'].astype(str).str.strip().str.upper() == 'CAD').astype(int).values,
    'LAD': (df_raw['LAD'].astype(str).str.strip().str.capitalize() == 'Stenotic').astype(int).values,
    'LCX': (df_raw['LCX'].astype(str).str.strip().str.capitalize() == 'Stenotic').astype(int).values,
    'RCA': (df_raw['RCA'].astype(str).str.strip().str.capitalize() == 'Stenotic').astype(int).values,
}

print(f"Domain Feature Engineering: 6 physiological biomarkers added.")
print(f"Leakage columns strictly excluded: {leakage_cols}")
print(f"Input feature space shape: {X_df.shape}")


Domain Feature Engineering: 6 physiological biomarkers added.
Leakage columns strictly excluded: ['LAD', 'LCX', 'RCA', 'Cath']
Input feature space shape: (303, 61)


In [4]:
def build_feature_preprocessor(df: pd.DataFrame):
    leakage = ['LAD', 'LCX', 'RCA', 'Cath']
    candidate = [c for c in df.columns if c not in leakage]
    valid = [c for c in candidate if df[c].nunique() > 1]
    
    num_cols = [c for c in valid if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() > 5]
    cat_cols = [c for c in valid if c not in num_cols]
    
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    preprocessor = ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols)
    ], remainder='drop')
    return preprocessor, valid, num_cols, cat_cols

preprocessor, valid_features, num_cols, cat_cols = build_feature_preprocessor(df_engineered)
print(f"Continuous numerical features : {len(num_cols)}")
print(f"Categorical features          : {len(cat_cols)}")
print(f"Total input features          : {len(valid_features)}")


Continuous numerical features : 27
Categorical features          : 33
Total input features          : 60


In [5]:
class CardiacRiskMLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_pytorch_mlp(X_tr, y_tr, X_val, y_val, device, epochs=60, lr=0.003):
    input_dim = X_tr.shape[1]
    model = CardiacRiskMLP(input_dim).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    
    tr_dataset = TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.float32))
    tr_loader = DataLoader(tr_dataset, batch_size=32, shuffle=True)
    val_x = torch.tensor(X_val, dtype=torch.float32).to(device)
    val_y = torch.tensor(y_val, dtype=torch.float32).to(device)
    
    best_loss = float('inf')
    best_preds = None
    
    for epoch in range(epochs):
        model.train()
        for bx, by in tr_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            out = model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            
        model.eval()
        with torch.no_grad():
            vout = model(val_x)
            vloss = criterion(vout, val_y).item()
            if vloss < best_loss:
                best_loss = vloss
                best_preds = torch.sigmoid(vout).cpu().numpy()
                
    return best_preds

print("CardiacRiskMLP architecture and training routines defined.")


CardiacRiskMLP architecture and training routines defined.


In [6]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

model_factories = {
    "Balanced_ExtraTrees": lambda: ExtraTreesClassifier(n_estimators=180, max_depth=6, min_samples_split=4, class_weight='balanced', random_state=SEED),
    "Balanced_RandomForest": lambda: RandomForestClassifier(n_estimators=180, max_depth=6, min_samples_split=4, class_weight='balanced', random_state=SEED),
    "HistGradientBoosting": lambda: HistGradientBoostingClassifier(max_iter=120, max_depth=4, learning_rate=0.04, random_state=SEED),
    "LogisticRegression": lambda: LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced', random_state=SEED),
    "Orthogonal_Soft_Voting": lambda: VotingClassifier(
        estimators=[
            ('et', ExtraTreesClassifier(n_estimators=180, max_depth=6, min_samples_split=4, class_weight='balanced', random_state=SEED)),
            ('rf', RandomForestClassifier(n_estimators=180, max_depth=6, min_samples_split=4, class_weight='balanced', random_state=SEED)),
            ('lr', LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced', random_state=SEED))
        ], voting='soft', weights=[0.45, 0.40, 0.15]
    ),
    "PyTorch_Deep_MLP": "pytorch_mlp"
}

all_cv_results = {}

for target_name, y in targets.items():
    print(f"\n==========================================================================================")
    print(f"  5-FOLD STRATIFIED BENCHMARK: {target_name} ({np.mean(y)*100:.1f}% Positive Class)")
    print(f"==========================================================================================")
    
    results = {m: {
        "accuracy": [], "precision": [], "recall": [], "specificity": [], "f1": [], "f2": [], "roc_auc": [], "pr_auc": [], "brier": [],
        "y_true_all": [], "y_prob_all": [], "conf_matrix_sum": np.zeros((2, 2), dtype=int)
    } for m in model_factories}
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_df, y)):
        X_tr_df, X_va_df = X_df.iloc[train_idx], X_df.iloc[val_idx]
        y_tr, y_va = y[train_idx], y[val_idx]
        
        prep = clone(preprocessor)
        X_tr_trans = prep.fit_transform(X_tr_df)
        X_va_trans = prep.transform(X_va_df)
        
        for m_name, m_ctor in model_factories.items():
            if m_name == "PyTorch_Deep_MLP":
                y_prob = train_pytorch_mlp(X_tr_trans, y_tr, X_va_trans, y_va, device)
            else:
                clf = m_ctor()
                clf.fit(X_tr_trans, y_tr)
                y_prob = clf.predict_proba(X_va_trans)[:, 1]
                
            y_pred = (y_prob >= 0.5).astype(int)
            acc = accuracy_score(y_va, y_pred)
            prec = precision_score(y_va, y_pred, zero_division=0)
            rec = recall_score(y_va, y_pred, zero_division=0)
            f1 = f1_score(y_va, y_pred, zero_division=0)
            f2 = fbeta_score(y_va, y_pred, beta=2, zero_division=0)
            try:
                auc = roc_auc_score(y_va, y_prob)
            except Exception:
                auc = 0.5
            try:
                pr_auc = average_precision_score(y_va, y_prob)
            except Exception:
                pr_auc = float(np.mean(y_va))
            brier = brier_score_loss(y_va, y_prob)
                
            tn, fp, fn, tp = confusion_matrix(y_va, y_pred, labels=[0, 1]).ravel()
            spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
            
            results[m_name]["accuracy"].append(acc)
            results[m_name]["precision"].append(prec)
            results[m_name]["recall"].append(rec)
            results[m_name]["specificity"].append(spec)
            results[m_name]["f1"].append(f1)
            results[m_name]["f2"].append(f2)
            results[m_name]["roc_auc"].append(auc)
            results[m_name]["pr_auc"].append(pr_auc)
            results[m_name]["brier"].append(brier)
            results[m_name]["y_true_all"].extend(y_va.tolist())
            results[m_name]["y_prob_all"].extend(y_prob.tolist())
            results[m_name]["conf_matrix_sum"] += np.array([[tn, fp], [fn, tp]])
            
    summary = {}
    for m_name, met in results.items():
        summary[m_name] = {
            "accuracy_mean": float(np.mean(met["accuracy"])),
            "accuracy_std": float(np.std(met["accuracy"])),
            "precision_mean": float(np.mean(met["precision"])),
            "precision_std": float(np.std(met["precision"])),
            "recall_mean": float(np.mean(met["recall"])),
            "recall_std": float(np.std(met["recall"])),
            "specificity_mean": float(np.mean(met["specificity"])),
            "specificity_std": float(np.std(met["specificity"])),
            "f1_mean": float(np.mean(met["f1"])),
            "f1_std": float(np.std(met["f1"])),
            "f2_mean": float(np.mean(met["f2"])),
            "f2_std": float(np.std(met["f2"])),
            "roc_auc_mean": float(np.mean(met["roc_auc"])),
            "roc_auc_std": float(np.std(met["roc_auc"])),
            "pr_auc_mean": float(np.mean(met["pr_auc"])),
            "pr_auc_std": float(np.std(met["pr_auc"])),
            "brier_mean": float(np.mean(met["brier"])),
            "brier_std": float(np.std(met["brier"])),
            "confusion_matrix": met["conf_matrix_sum"].tolist()
        }
        print(f"  {m_name:<22} | ROC-AUC: {summary[m_name]['roc_auc_mean']:.4f} +/- {summary[m_name]['roc_auc_std']:.3f} | PR-AUC: {summary[m_name]['pr_auc_mean']:.4f} | F2: {summary[m_name]['f2_mean']:.4f} | F1: {summary[m_name]['f1_mean']:.4f} | Brier: {summary[m_name]['brier_mean']:.4f}")
    all_cv_results[target_name] = summary



  5-FOLD STRATIFIED BENCHMARK: CAD (71.3% Positive Class)
  Balanced_ExtraTrees    | ROC-AUC: 0.9252 +/- 0.035 | PR-AUC: 0.9707 | F2: 0.8868 | F1: 0.8980 | Brier: 0.1100
  Balanced_RandomForest  | ROC-AUC: 0.9202 +/- 0.049 | PR-AUC: 0.9667 | F2: 0.9069 | F1: 0.8998 | Brier: 0.1099
  HistGradientBoosting   | ROC-AUC: 0.9068 +/- 0.040 | PR-AUC: 0.9603 | F2: 0.9141 | F1: 0.8975 | Brier: 0.1124
  LogisticRegression     | ROC-AUC: 0.9325 +/- 0.033 | PR-AUC: 0.9727 | F2: 0.8745 | F1: 0.8959 | Brier: 0.1013
  Orthogonal_Soft_Voting | ROC-AUC: 0.9300 +/- 0.038 | PR-AUC: 0.9721 | F2: 0.8944 | F1: 0.9030 | Brier: 0.1024
  PyTorch_Deep_MLP       | ROC-AUC: 0.9254 +/- 0.028 | PR-AUC: 0.9662 | F2: 0.9039 | F1: 0.9065 | Brier: 0.0978

  5-FOLD STRATIFIED BENCHMARK: LAD (58.4% Positive Class)
  Balanced_ExtraTrees    | ROC-AUC: 0.8441 +/- 0.057 | PR-AUC: 0.8801 | F2: 0.8383 | F1: 0.8248 | Brier: 0.1599
  Balanced_RandomForest  | ROC-AUC: 0.8618 +/- 0.056 | PR-AUC: 0.8942 | F2: 0.8465 | F1: 0.8288 | 

In [7]:
production_model_defs = {
    "CAD": VotingClassifier(
        estimators=[
            ('et', ExtraTreesClassifier(n_estimators=180, max_depth=6, min_samples_split=4, class_weight='balanced', random_state=SEED)),
            ('rf', RandomForestClassifier(n_estimators=180, max_depth=6, min_samples_split=4, class_weight='balanced', random_state=SEED)),
            ('lr', LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced', random_state=SEED))
        ], voting='soft', weights=[0.45, 0.40, 0.15]
    ),
    "LAD": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=180, max_depth=6, min_samples_split=4, class_weight='balanced', random_state=SEED)),
            ('et', ExtraTreesClassifier(n_estimators=180, max_depth=6, min_samples_split=4, class_weight='balanced', random_state=SEED)),
            ('hgb', HistGradientBoostingClassifier(max_iter=100, max_depth=4, learning_rate=0.04, random_state=SEED))
        ], voting='soft', weights=[0.50, 0.35, 0.15]
    ),
    "LCX": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=180, max_depth=5, min_samples_split=6, class_weight='balanced', random_state=SEED)),
            ('et', ExtraTreesClassifier(n_estimators=180, max_depth=5, min_samples_split=6, class_weight='balanced', random_state=SEED)),
            ('hgb', HistGradientBoostingClassifier(max_iter=100, max_depth=3, learning_rate=0.03, random_state=SEED))
        ], voting='soft', weights=[0.45, 0.40, 0.15]
    ),
    "RCA": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=180, max_depth=5, min_samples_split=6, class_weight='balanced', random_state=SEED)),
            ('et', ExtraTreesClassifier(n_estimators=180, max_depth=5, min_samples_split=6, class_weight='balanced', random_state=SEED)),
            ('lr', LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced', random_state=SEED))
        ], voting='soft', weights=[0.50, 0.35, 0.15]
    ),
}

clinical_thresholds = {
    "CAD": 0.48,
    "LAD": 0.46,
    "LCX": 0.42,
    "RCA": 0.44
}

full_prep = clone(preprocessor)
X_full_trans = full_prep.fit_transform(X_df)

try:
    feature_names = full_prep.get_feature_names_out()
    clean_feature_names = [f.replace("num__", "").replace("cat__", "") for f in feature_names]
except Exception:
    clean_feature_names = [f"Feature_{i}" for i in range(X_full_trans.shape[1])]

trained_models = {}
explainers = {}

for target_name, clf in production_model_defs.items():
    clf.fit(X_full_trans, targets[target_name])
    trained_models[target_name] = clf
    primary_tree = clf.named_estimators_['rf'] if 'rf' in clf.named_estimators_ else clf.named_estimators_['et']
    explainers[target_name] = shap.TreeExplainer(primary_tree)
    print(f"Trained production model for {target_name}: {clf.__class__.__name__} (Soft Voting)")

os.makedirs("models", exist_ok=True)
pipeline_artifact = {
    "preprocessor": full_prep,
    "models": trained_models,
    "explainers": explainers,
    "feature_names": clean_feature_names,
    "input_features": valid_features,
    "raw_input_features": [c for c in df_raw.columns if c not in leakage_cols and c != 'Exertional CP'],
    "numerical_features": num_cols,
    "categorical_features": cat_cols,
    "target_names": ["CAD", "LAD", "LCX", "RCA"],
    "clinical_thresholds": clinical_thresholds
}

save_path = "models/cardioscope_pipeline.joblib"
joblib.dump(pipeline_artifact, save_path)
print(f"\nPipeline artifact successfully saved to: {save_path}")


Trained production model for CAD: VotingClassifier (Soft Voting)
Trained production model for LAD: VotingClassifier (Soft Voting)
Trained production model for LCX: VotingClassifier (Soft Voting)
Trained production model for RCA: VotingClassifier (Soft Voting)

Pipeline artifact successfully saved to: models/cardioscope_pipeline.joblib


In [8]:
print("=" * 70)
print("TREESHAP GLOBAL FEATURE ATTRIBUTION RANKINGS")
print("=" * 70)

for target_name, clf in trained_models.items():
    explainer = explainers[target_name]
    shap_vals = explainer.shap_values(X_full_trans)
    
    if isinstance(shap_vals, list):
        sv = shap_vals[1]
    elif len(shap_vals.shape) == 3:
        sv = shap_vals[:, :, 1]
    else:
        sv = shap_vals
        
    mean_abs = np.mean(np.abs(sv), axis=0)
    top_indices = np.argsort(mean_abs)[::-1][:10]
    
    print(f"\nTop Influential Predictors for {target_name}:")
    for rank, idx in enumerate(top_indices, 1):
        feat_name = clean_feature_names[idx]
        imp_val = float(mean_abs[idx])
        print(f"  {rank:2d}. {feat_name:<30} | Mean |SHAP|: {imp_val:.4f}")


TREESHAP GLOBAL FEATURE ATTRIBUTION RANKINGS

Top Influential Predictors for CAD:
   1. Ischemic_Burden                | Mean |SHAP|: 0.0605
   2. Typical Chest Pain_0           | Mean |SHAP|: 0.0550
   3. Typical Chest Pain_1           | Mean |SHAP|: 0.0504
   4. Ventricular_Perf_Index         | Mean |SHAP|: 0.0282
   5. Age                            | Mean |SHAP|: 0.0276
   6. Atypical_N                     | Mean |SHAP|: 0.0263
   7. EF-TTE                         | Mean |SHAP|: 0.0236
   8. Atypical_Y                     | Mean |SHAP|: 0.0198
   9. Region RWMA_0                  | Mean |SHAP|: 0.0192
  10. Inflammatory_Age_Index         | Mean |SHAP|: 0.0173

Top Influential Predictors for LAD:
   1. Ischemic_Burden                | Mean |SHAP|: 0.0599
   2. Ventricular_Perf_Index         | Mean |SHAP|: 0.0359
   3. Typical Chest Pain_1           | Mean |SHAP|: 0.0346
   4. Typical Chest Pain_0           | Mean |SHAP|: 0.0326
   5. EF-TTE                         | Mean |SHAP|: 0.0

In [9]:
print("=" * 75)
print("CLINICAL PATIENT PROFILES PREDICTION VERIFICATION")
print("=" * 75)

sample_row_indices = [0, 3, 5, 9]
for idx in sample_row_indices:
    sample_raw = df_raw.iloc[[idx]].copy()
    sample_eng = add_domain_features(sample_raw)
    sample_X = sample_eng.drop(columns=leakage_cols, errors='ignore')
    sample_trans = full_prep.transform(sample_X)
    
    print(f"\nPatient Row {idx + 1}:")
    print(f"  Age: {sample_raw['Age'].values[0]} | Sex: {sample_raw['Sex'].values[0]} | BP: {sample_raw['BP'].values[0]} | PR: {sample_raw['PR'].values[0]} | Chest Pain: {sample_raw['Typical Chest Pain'].values[0]}")
    print(f"  Ground Truth -> CAD: {sample_raw['Cath'].values[0]} | LAD: {sample_raw['LAD'].values[0]} | LCX: {sample_raw['LCX'].values[0]} | RCA: {sample_raw['RCA'].values[0]}")
    
    pred_strings = []
    for t_name, clf in trained_models.items():
        prob = clf.predict_proba(sample_trans)[0, 1]
        thresh = clinical_thresholds.get(t_name, 0.5)
        status = "POSITIVE" if prob >= thresh else "NEGATIVE"
        pred_strings.append(f"{t_name}: {prob*100:5.1f}% ({status})")
    print("  Predictions  -> " + " | ".join(pred_strings))


CLINICAL PATIENT PROFILES PREDICTION VERIFICATION

Patient Row 1:
  Age: 53 | Sex: Male | BP: 110 | PR: 80 | Chest Pain: 0
  Ground Truth -> CAD: CAD | LAD: Stenotic | LCX: Normal | RCA: Stenotic
  Predictions  -> CAD:  70.9% (POSITIVE) | LAD:  63.0% (POSITIVE) | LCX:  30.4% (NEGATIVE) | RCA:  50.8% (POSITIVE)

Patient Row 4:
  Age: 66 | Sex: Fmale | BP: 100 | PR: 80 | Chest Pain: 0
  Ground Truth -> CAD: Normal | LAD: Normal | LCX: Normal | RCA: Normal
  Predictions  -> CAD:  17.5% (NEGATIVE) | LAD:  17.8% (NEGATIVE) | LCX:  25.0% (NEGATIVE) | RCA:  23.9% (NEGATIVE)

Patient Row 6:
  Age: 50 | Sex: Male | BP: 118 | PR: 70 | Chest Pain: 1
  Ground Truth -> CAD: CAD | LAD: Stenotic | LCX: Stenotic | RCA: Stenotic
  Predictions  -> CAD:  73.2% (POSITIVE) | LAD:  72.7% (POSITIVE) | LCX:  61.1% (POSITIVE) | RCA:  69.2% (POSITIVE)

Patient Row 10:
  Age: 60 | Sex: Male | BP: 130 | PR: 70 | Chest Pain: 1
  Ground Truth -> CAD: CAD | LAD: Normal | LCX: Stenotic | RCA: Stenotic
  Predictions  

In [10]:
summary_rows = []
for target in ["CAD", "LAD", "LCX", "RCA"]:
    for m_name in model_factories:
        m_stats = all_cv_results[target][m_name]
        summary_rows.append({
            "Target": target,
            "Model": m_name,
            "ROC-AUC": f"{m_stats['roc_auc_mean']:.4f} +/- {m_stats['roc_auc_std']:.3f}",
            "PR-AUC": f"{m_stats['pr_auc_mean']:.4f}",
            "F2-Score": f"{m_stats['f2_mean']:.4f}",
            "F1-Score": f"{m_stats['f1_mean']:.4f}",
            "Accuracy": f"{m_stats['accuracy_mean']:.4f}",
            "Sensitivity": f"{m_stats['recall_mean']:.4f}",
            "Specificity": f"{m_stats['specificity_mean']:.4f}",
            "Brier": f"{m_stats['brier_mean']:.4f}",
        })

df_summary = pd.DataFrame(summary_rows)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print("\n" + "=" * 115)
print("COMPREHENSIVE 5-FOLD STRATIFIED BENCHMARK TABLE (ADVANCED CLINICAL METRICS)")
print("=" * 115)
print(df_summary.to_string(index=False))



COMPREHENSIVE 5-FOLD STRATIFIED BENCHMARK TABLE (ADVANCED CLINICAL METRICS)
Target                  Model          ROC-AUC PR-AUC F2-Score F1-Score Accuracy Sensitivity Specificity  Brier
   CAD    Balanced_ExtraTrees 0.9252 +/- 0.035 0.9707   0.8868   0.8980   0.8579      0.8798      0.8039 0.1100
   CAD  Balanced_RandomForest 0.9202 +/- 0.049 0.9667   0.9069   0.8998   0.8546      0.9121      0.7144 0.1099
   CAD   HistGradientBoosting 0.9068 +/- 0.040 0.9603   0.9141   0.8975   0.8481      0.9259      0.6556 0.1124
   CAD     LogisticRegression 0.9325 +/- 0.033 0.9727   0.8745   0.8959   0.8577      0.8610      0.8497 0.1013
   CAD Orthogonal_Soft_Voting 0.9300 +/- 0.038 0.9721   0.8944   0.9030   0.8644      0.8890      0.8046 0.1024
   CAD       PyTorch_Deep_MLP 0.9254 +/- 0.028 0.9662   0.9039   0.9065   0.8677      0.9026      0.7824 0.0978
   LAD    Balanced_ExtraTrees 0.8441 +/- 0.057 0.8801   0.8383   0.8248   0.7891      0.8483      0.7065 0.1599
   LAD  Balanced_RandomFore